## Exercise
## A more realistic Peter and the Wolf world

1. By moving from one place to another, Peter loses energy and gains some fatigue.
2. Peter can gain more energy by eating apples.
3. Peter can get rid of fatigue by resting under the tree or on the grass (i.e. walking into a board location with a tree or grass - green field)
4. Peter needs to find and kill the wolf
5. In order to kill the wolf, Peter needs to have certain levels of energy and fatigue, otherwise he loses the battle.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
from rlboard import *

# --- Actions (keys used by Q-learning + UI) ---
actions = {"U": (0, -1), "D": (0, 1), "L": (-1, 0), "R": (1, 0)}
action_list = list(actions.keys())
action_idx = {a: i for i, a in enumerate(action_list)}

# --- Cell constants from your rlboard.py ---
WATER = Board.Cell.water
APPLE = Board.Cell.apple
WOLF  = Board.Cell.wolf
TREE  = Board.Cell.tree
EMPTY = Board.Cell.empty

REST_CELLS = {TREE}  # your rlboard has ONLY tree as green/rest

def is_rest_cell(cell):
    return cell in REST_CELLS


class PeterWolfEnv:
    """
    Environment wrapper around rlboard.Board + extra state:
    - Peter position
    - energy, fatigue
    - apples disappear when eaten
    - wolf alive
    - peter alive
    """

    def __init__(
        self,
        board: Board,
        energy_start=100,
        fatigue_start=0,
        energy_min=0,
        energy_max=120,
        fatigue_min=0,
        fatigue_max=120,
        move_energy_cost=2,
        move_fatigue_gain=1,
        apple_energy_gain=25,
        rest_fatigue_recover=15,
        fight_energy_need=70,
        fight_fatigue_max=60,
    ):
        self.b = board
        self.width = board.width
        self.height = board.height

        # params
        self.energy_start = energy_start
        self.fatigue_start = fatigue_start
        self.energy_min = energy_min
        self.energy_max = energy_max
        self.fatigue_min = fatigue_min
        self.fatigue_max = fatigue_max

        self.move_energy_cost = move_energy_cost
        self.move_fatigue_gain = move_fatigue_gain
        self.apple_energy_gain = apple_energy_gain
        self.rest_fatigue_recover = rest_fatigue_recover

        self.fight_energy_need = fight_energy_need
        self.fight_fatigue_max = fight_fatigue_max

        # dynamic state
        self.pos = None
        self.energy = None
        self.fatigue = None
        self.wolf_alive = True
        self.peter_alive = True
        self.apples = set()
        self.wolf_pos = None

        # IMPORTANT: keep an original copy so reset() can restore apples
        self._orig_matrix = self.b.matrix.copy()

        self.reset(random_start=True)

    def reset(self, random_start=True, start_pos=None):
        # restore board exactly (apples come back)
        self.b.matrix = self._orig_matrix.copy()

        # scan board for apples + wolf
        self.apples.clear()
        self.wolf_pos = None
        for x in range(self.width):
            for y in range(self.height):
                c = self.b.at((x, y))
                if c == APPLE:
                    self.apples.add((x, y))
                elif c == WOLF:
                    self.wolf_pos = (x, y)

        self.energy = self.energy_start
        self.fatigue = self.fatigue_start
        self.wolf_alive = True
        self.peter_alive = True

        if start_pos is not None:
            self.pos = start_pos
        else:
            if random_start:
                self.b.random_start()
                self.pos = self.b.human
            else:
                self.pos = (0, 0)

        self.b.human = self.pos
        return self.state_key()

    def clamp(self):
        self.energy = max(self.energy_min, min(self.energy, self.energy_max))
        self.fatigue = max(self.fatigue_min, min(self.fatigue, self.fatigue_max))

    def discretize(self, value, step=10):
        return int(value // step)

    def state_key(self):
        """
        Keep the state small so Q-learning has a chance.
        If you include exact apple positions, state space explodes.
        """
        return (
            self.pos[0], self.pos[1],
            self.discretize(self.energy, 10),
            self.discretize(self.fatigue, 10),
            int(self.wolf_alive),
            len(self.apples),  # compact instead of tuple(sorted(self.apples))
        )

    def done(self):
        return (not self.peter_alive) or (not self.wolf_alive)

    def step(self, action_key):
        if self.done():
            return self.state_key(), 0.0, True

        dx, dy = actions[action_key]
        nx, ny = self.pos[0] + dx, self.pos[1] + dy
        next_pos = (nx, ny)

        # movement costs happen on attempt
        self.energy -= self.move_energy_cost
        self.fatigue += self.move_fatigue_gain
        self.clamp()

        # invalid move => lose
        if not self.b.is_valid(next_pos):
            self.peter_alive = False
            return self.state_key(), -10.0, True

        cell = self.b.at(next_pos)

        # water => lose
        if cell == WATER:
            self.pos = next_pos
            self.b.human = self.pos
            self.peter_alive = False
            return self.state_key(), -10.0, True

        # move succeeds
        self.pos = next_pos
        self.b.human = self.pos

        reward = -0.1

        # apple => energy gain + apple disappears (IN SET + IN BOARD)
        if next_pos in self.apples:
            self.apples.remove(next_pos)

            # remove apple from the board matrix so it disappears visually / logically
            self.b.matrix[next_pos[0], next_pos[1]] = EMPTY

            self.energy += self.apple_energy_gain
            self.clamp()
            reward += 1.0

        # rest cell => fatigue recover
        if is_rest_cell(cell):
            self.fatigue -= self.rest_fatigue_recover
            self.clamp()
            reward += 0.5

        # wolf encounter
        if self.wolf_alive and self.wolf_pos is not None and self.pos == self.wolf_pos:
            if self.energy >= self.fight_energy_need and self.fatigue <= self.fight_fatigue_max:
                self.wolf_alive = False
                reward += 10.0
                return self.state_key(), reward, True
            else:
                self.peter_alive = False
                reward += -10.0
                return self.state_key(), reward, True

        # optional: energy depletion ends episode
        if self.energy <= 0:
            self.peter_alive = False
            reward += -10.0
            return self.state_key(), reward, True

        return self.state_key(), reward, False


def random_policy(_env: PeterWolfEnv):
    return random.choice(action_list)


def greedy_policy_from_Q(env: PeterWolfEnv, Q):
    key = env.state_key()
    if key not in Q:
        return random.choice(action_list)
    return action_list[int(np.argmax(Q[key]))]


def walk_episode(env: PeterWolfEnv, policy_fn, max_steps=10):
    env.reset(random_start=True)
    total = 0.0
    for step in range(max_steps):
        a = policy_fn(env)
        _, r, done = env.step(a)
        total += r
        if done:
            return (not env.wolf_alive), (step + 1), total
    return False, max_steps, total


def train_q_learning(
    env: PeterWolfEnv,
    epochs=3,
    gamma=0.95,
    alpha_start=0.4,
    alpha_end=0.05,
    eps_start=1.0,
    eps_end=0.05,
    max_steps=10
):
    Q = {}

    def get_Q(state_key):
        if state_key not in Q:
            Q[state_key] = np.zeros(len(action_list), dtype=float)
        return Q[state_key]

    for ep in range(epochs):
        env.reset(random_start=True)

        frac = ep / max(1, epochs - 1)
        alpha = alpha_start * (1 - frac) + alpha_end * frac
        eps = eps_start * (1 - frac) + eps_end * frac

        state = env.state_key()

        for _ in range(max_steps):
            if random.random() < eps:
                a = random.choice(action_list)
            else:
                a = action_list[int(np.argmax(get_Q(state)))]

            next_state, r, done = env.step(a)

            qsa = get_Q(state)[action_idx[a]]
            best_next = 0.0 if done else float(np.max(get_Q(next_state)))

            get_Q(state)[action_idx[a]] = (1 - alpha) * qsa + alpha * (r + gamma * best_next)

            state = next_state
            if done:
                break

    return Q


def evaluate(env, Q=None, episodes=200):
    wins = 0
    losses = 0
    steps_list = []

    for _ in range(episodes):
        if Q is None:
            policy = random_policy
        else:
            policy = lambda e: greedy_policy_from_Q(e, Q)

        won, steps, _ = walk_episode(env, policy_fn=policy, max_steps=10)
        if won:
            wins += 1
        else:
            losses += 1
        steps_list.append(steps)

    return {"wins": wins, "losses": losses, "avg_steps": sum(steps_list) / len(steps_list)}




# Evaluation

In [9]:
width, height = 8, 8
m = Board(width, height)
m.randomize(seed=13)

env = PeterWolfEnv(m)

print("Random:", evaluate(env, Q=None, episodes=200))

Q = train_q_learning(env, epochs=20000)

print("Learned:", evaluate(env, Q=Q, episodes=200))

Random: {'wins': 3, 'losses': 197, 'avg_steps': 4.095}
Learned: {'wins': 123, 'losses': 77, 'avg_steps': 7.605}
